## 05 — Business Analytics Queries

Answering key business questions using the **star schema** defined in `04-Star-Schema-Definition`.

All queries join through the fact and dimension tables:

| Table | Role |
| --- | --- |
| `fact_violation` | Central fact — one row per violation event |
| `dim_business` | Business identity, facility type, risk |
| `dim_location` | Address, city, state, zip, lat/lon |
| `dim_date` | Date breakdown (year, month, quarter, day-of-week) |
| `dim_inspection_type` | Type of inspection performed |
| `dim_violation_type` | Violation code and description |

| # | Question |
| --- | --- |
| 1 | Which violation categories are most strongly associated with failed inspections? |
| 2 | What are the worst-performing ZIP codes over time? |
| 3 | Are complaint-based inspections more likely to fail than scheduled inspections? |
| 4 | Which businesses repeatedly fail inspections over multiple years? |
| 5 | Which ZIP codes have the highest failure rates? |
| 6 | What are the most common violations? Are they trending up or down? |
| 7 | Does risk correlate with inspection outcomes? |
| 8 | How long does it take a facility that failed to pass a reinspection? |

In [0]:
# Table references — gold layer star schema
SCHEMA = "students_data.`team4-chicago`"

FACT_VIOLATION     = f"{SCHEMA}.fact_violation"
DIM_BUSINESS       = f"{SCHEMA}.dim_business"
DIM_LOCATION       = f"{SCHEMA}.dim_location"
DIM_DATE           = f"{SCHEMA}.dim_date"
DIM_INSPECTION     = f"{SCHEMA}.dim_inspection_type"
DIM_VIOLATION_TYPE = f"{SCHEMA}.dim_violation_type"

In [0]:
# Q1: Which violation categories are most strongly associated with failed inspections?
# Join fact → dim_violation_type to get violation descriptions, group by violation code

q1 = spark.sql(f"""
SELECT
    vt.violation_code,
    vt.violation_description,
    COUNT(*) AS total_occurrences,
    SUM(CASE WHEN f.result = 'Fail' THEN 1 ELSE 0 END) AS fail_count,
    ROUND(SUM(CASE WHEN f.result = 'Fail' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS fail_rate_pct
FROM {FACT_VIOLATION} f
JOIN {DIM_VIOLATION_TYPE} vt ON f.violation_type_key = vt.violation_type_key
GROUP BY vt.violation_code, vt.violation_description
HAVING COUNT(*) >= 100
ORDER BY fail_rate_pct DESC
""")

display(q1)

In [0]:
# Q2: What are the worst-performing ZIP codes over time?
# Join fact → dim_location (zip) and dim_date (year), fail rate per zip per year
# Use COUNT(DISTINCT inspection_id) — fact_violation grain is per-violation, not per-inspection

q2 = spark.sql(f"""
SELECT
    l.zip,
    d.year,
    COUNT(DISTINCT f.inspection_id) AS total_inspections,
    COUNT(DISTINCT CASE WHEN f.result = 'Fail' THEN f.inspection_id END) AS failures,
    ROUND(COUNT(DISTINCT CASE WHEN f.result = 'Fail' THEN f.inspection_id END) * 100.0
          / COUNT(DISTINCT f.inspection_id), 1) AS fail_rate_pct
FROM {FACT_VIOLATION} f
JOIN {DIM_LOCATION} l ON f.location_key = l.location_key
JOIN {DIM_DATE} d ON f.date_key = d.date_key
GROUP BY l.zip, d.year
HAVING COUNT(DISTINCT f.inspection_id) >= 50
ORDER BY fail_rate_pct DESC, d.year DESC
""")

display(q2)

In [0]:
# Q3: Are complaint-based inspections more likely to fail than scheduled inspections?
# Join fact → dim_inspection_type, compare fail rates by inspection category
# Use COUNT(DISTINCT inspection_id) and group types into Complaint vs Routine categories

q3 = spark.sql(f"""
SELECT
    CASE
        WHEN it.inspection_type IN ('Complaint', 'Complaint-Fire', 'Short Form Complaint',
             'Short Form Fire-Complaint', 'Suspected Food Poisoning') THEN 'Complaint-Based'
        WHEN it.inspection_type IN ('Canvas', 'License') THEN 'Routine / Scheduled'
        WHEN LOWER(it.inspection_type) LIKE '%re-inspection%' THEN 'Re-Inspection'
        ELSE 'Other'
    END AS inspection_category,
    it.inspection_type,
    COUNT(DISTINCT f.inspection_id) AS total_inspections,
    COUNT(DISTINCT CASE WHEN f.result = 'Fail' THEN f.inspection_id END) AS failures,
    COUNT(DISTINCT CASE WHEN f.result = 'Pass' THEN f.inspection_id END) AS passes,
    COUNT(DISTINCT CASE WHEN f.result = 'Pass w/ Conditions' THEN f.inspection_id END) AS conditional_passes,
    ROUND(COUNT(DISTINCT CASE WHEN f.result = 'Fail' THEN f.inspection_id END) * 100.0
          / COUNT(DISTINCT f.inspection_id), 1) AS fail_rate_pct
FROM {FACT_VIOLATION} f
JOIN {DIM_INSPECTION} it ON f.inspection_type_key = it.inspection_type_key
GROUP BY inspection_category, it.inspection_type
ORDER BY inspection_category, fail_rate_pct DESC
""")

display(q3)

In [0]:
# Q4: Which businesses repeatedly fail inspections over multiple years?
# Join fact → dim_business + dim_date, find businesses failing in 3+ distinct years
# Use COUNT(DISTINCT inspection_id) — fact_violation grain is per-violation, not per-inspection

q4 = spark.sql(f"""
SELECT
    b.license_number,
    b.dba_name,
    b.facility_type,
    b.risk,
    COUNT(DISTINCT d.year) AS years_with_failures,
    COUNT(DISTINCT f.inspection_id) AS total_failed_inspections,
    MIN(d.full_date) AS first_failure,
    MAX(d.full_date) AS latest_failure
FROM {FACT_VIOLATION} f
JOIN {DIM_BUSINESS} b ON f.business_key = b.business_key
JOIN {DIM_DATE} d ON f.date_key = d.date_key
WHERE f.result = 'Fail'
GROUP BY b.license_number, b.dba_name, b.facility_type, b.risk
HAVING COUNT(DISTINCT d.year) >= 3
ORDER BY years_with_failures DESC, total_failed_inspections DESC
""")

display(q4)

In [0]:
# Q5: Which ZIP codes have the highest failure rates?
# Join fact → dim_location, overall fail rate per ZIP with 100+ inspections
# Use COUNT(DISTINCT inspection_id) — fact_violation grain is per-violation, not per-inspection

q5 = spark.sql(f"""
SELECT
    l.zip,
    COUNT(DISTINCT f.inspection_id) AS total_inspections,
    COUNT(DISTINCT CASE WHEN f.result = 'Fail' THEN f.inspection_id END) AS failures,
    COUNT(DISTINCT CASE WHEN f.result = 'Pass' THEN f.inspection_id END) AS passes,
    ROUND(COUNT(DISTINCT CASE WHEN f.result = 'Fail' THEN f.inspection_id END) * 100.0
          / COUNT(DISTINCT f.inspection_id), 1) AS fail_rate_pct,
    ROUND(COUNT(DISTINCT CASE WHEN f.result = 'Pass' THEN f.inspection_id END) * 100.0
          / COUNT(DISTINCT f.inspection_id), 1) AS pass_rate_pct
FROM {FACT_VIOLATION} f
JOIN {DIM_LOCATION} l ON f.location_key = l.location_key
GROUP BY l.zip
HAVING COUNT(DISTINCT f.inspection_id) >= 100
ORDER BY fail_rate_pct DESC
""")

display(q5)

In [0]:
# Q6: What are the most common violations? Are they trending up or down?
# Join fact → dim_violation_type + dim_date, top 10 violations with year-over-year counts

q6 = spark.sql(f"""
WITH top_codes AS (
    SELECT vt.violation_code
    FROM {FACT_VIOLATION} f
    JOIN {DIM_VIOLATION_TYPE} vt ON f.violation_type_key = vt.violation_type_key
    GROUP BY vt.violation_code
    ORDER BY COUNT(*) DESC
    LIMIT 10
)
SELECT
    vt.violation_code,
    vt.violation_description,
    d.year,
    COUNT(*) AS occurrences
FROM {FACT_VIOLATION} f
JOIN {DIM_VIOLATION_TYPE} vt ON f.violation_type_key = vt.violation_type_key
JOIN {DIM_DATE} d ON f.date_key = d.date_key
WHERE vt.violation_code IN (SELECT violation_code FROM top_codes)
GROUP BY vt.violation_code, vt.violation_description, d.year
ORDER BY vt.violation_code, d.year
""")

display(q6)

In [0]:
# Q7: Does risk level correlate with inspection outcomes?
# Join fact → dim_business, cross-tabulation of risk vs result
# Use COUNT(DISTINCT inspection_id) — fact_violation grain is per-violation, not per-inspection

q7 = spark.sql(f"""
SELECT
    b.risk AS risk_level,
    COUNT(DISTINCT f.inspection_id) AS total_inspections,
    COUNT(DISTINCT CASE WHEN f.result = 'Fail' THEN f.inspection_id END) AS failures,
    COUNT(DISTINCT CASE WHEN f.result = 'Pass' THEN f.inspection_id END) AS passes,
    COUNT(DISTINCT CASE WHEN f.result = 'Pass w/ Conditions' THEN f.inspection_id END) AS conditional_passes,
    ROUND(COUNT(DISTINCT CASE WHEN f.result = 'Fail' THEN f.inspection_id END) * 100.0
          / COUNT(DISTINCT f.inspection_id), 1) AS fail_rate_pct,
    ROUND(COUNT(DISTINCT CASE WHEN f.result = 'Pass' THEN f.inspection_id END) * 100.0
          / COUNT(DISTINCT f.inspection_id), 1) AS pass_rate_pct
FROM {FACT_VIOLATION} f
JOIN {DIM_BUSINESS} b ON f.business_key = b.business_key
GROUP BY b.risk
ORDER BY fail_rate_pct DESC
""")

display(q7)

In [0]:
# Q8: How long does it take a facility that failed to pass a reinspection?
# Self-join fact_violation: find fail → next re-inspection pass for the same business
# Use dim_date for date arithmetic and dim_inspection_type to identify re-inspections

q8 = spark.sql(f"""
WITH failures AS (
    SELECT
        f.inspection_id,
        f.business_key,
        d.full_date AS fail_date
    FROM {FACT_VIOLATION} f
    JOIN {DIM_DATE} d ON f.date_key = d.date_key
    WHERE f.result = 'Fail'
),
reinspections AS (
    SELECT
        f.inspection_id,
        f.business_key,
        d.full_date AS reinspection_date,
        f.result AS reinspection_result
    FROM {FACT_VIOLATION} f
    JOIN {DIM_DATE} d ON f.date_key = d.date_key
    JOIN {DIM_INSPECTION} it ON f.inspection_type_key = it.inspection_type_key
    WHERE LOWER(it.inspection_type) LIKE '%re-inspection%'
       OR LOWER(it.inspection_type) LIKE '%reinspection%'
),
matched AS (
    SELECT
        fail.business_key,
        fail.fail_date,
        ri.reinspection_date,
        ri.reinspection_result,
        DATEDIFF(ri.reinspection_date, fail.fail_date) AS days_to_reinspection,
        ROW_NUMBER() OVER (
            PARTITION BY fail.business_key, fail.fail_date
            ORDER BY ri.reinspection_date
        ) AS rn
    FROM failures fail
    JOIN reinspections ri
        ON fail.business_key = ri.business_key
        AND ri.reinspection_date > fail.fail_date
        AND ri.reinspection_date <= DATE_ADD(fail.fail_date, 365)
)
SELECT
    reinspection_result,
    COUNT(*) AS total_reinspections,
    ROUND(AVG(days_to_reinspection), 1) AS avg_days,
    MIN(days_to_reinspection) AS min_days,
    MAX(days_to_reinspection) AS max_days,
    PERCENTILE_APPROX(days_to_reinspection, 0.5) AS median_days
FROM matched
WHERE rn = 1
GROUP BY reinspection_result
ORDER BY reinspection_result
""")

display(q8)